# Floor Plan Instance Segmentation & Dimension Analysis (YOLOv8-Seg)

This notebook is **100% self-contained** and runs out-of-the-box in Google Colab:
- **Primary Goal**: Detect all rooms and extract their real-world dimensions (length, width, area).
- **Secondary Goal**: Detect all doors (swings/openings) and windows (wall placements).
- **Core Technology**: YOLOv8-seg instance segmentation + Wall-snapping contour geometry + OCR dimension calibration.

> **Colab Runtime Recommendation**: Go to `Runtime` -> `Change runtime type` -> select **T4 GPU**.

## Step 1: Install System Dependencies
Installs Ultralytics YOLO, Tesseract OCR, and required geometry utilities.

In [ ]:
# 1. Install Python packages
!pip install -q "ultralytics>=8.3.0" "pytesseract>=0.3.10" "pyclipper>=1.3.0" "shapely>=2.0.0" "cairosvg>=2.7.0"

# 2. Install Tesseract OCR engine for reading blueprint dimension texts
!apt-get update -qq > /dev/null
!apt-get install -y -qq tesseract-ocr libtesseract-dev > /dev/null

import torch
import ultralytics
print(f"Ultralytics version: {ultralytics.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")


## Step 2: Self-Contained Package & Data Setup
Automatically unpacks and installs `floorplan_reader` and loads test blueprints.

In [ ]:
# ==============================================================================
# Step 2: Automatic Self-Contained Setup (Zero Git Clone or File Upload Needed)
# Unpacks the floorplan_reader package and test blueprints directly into Colab.
# ==============================================================================
import os, sys, base64, subprocess
from pathlib import Path

# 1. Unpack & Install floorplan_reader package
try:
    from floorplan_reader.segmentation.yolo_segmenter import YoloFloorPlanSegmenter
    from floorplan_reader.segmentation.dataset_exporter import YoloDatasetExporter
    from floorplan_reader.visualization.segmentation_visualizer import draw_styled_segmentation_overlay
    print("floorplan_reader package is already installed and ready!")
except ImportError:
    print("Installing floorplan_reader package from embedded wheel...")
    WHL_B64 = "..."
    whl_tmp = Path("/tmp/floorplan_reader-0.1.0-py3-none-any.whl")
    whl_tmp.write_bytes(base64.b64decode(WHL_B64))
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", str(whl_tmp)])
    
    from floorplan_reader.segmentation.yolo_segmenter import YoloFloorPlanSegmenter
    from floorplan_reader.segmentation.dataset_exporter import YoloDatasetExporter
    from floorplan_reader.visualization.segmentation_visualizer import draw_styled_segmentation_overlay
    print("Successfully installed and imported floorplan_reader!")

# 2. Extract Test Blueprint Samples
test_dir = Path("data/test_samples")
test_dir.mkdir(parents=True, exist_ok=True)

ms_path = test_dir / "sample_master_suite.png"
if not ms_path.exists():
    ms_path.write_bytes(base64.b64decode("..."))
    print("Extracted: data/test_samples/sample_master_suite.png")

b1_path = test_dir / "sample_1bhk.png"
if not b1_path.exists():
    b1_path.write_bytes(base64.b64decode("..."))
    print("Extracted: data/test_samples/sample_1bhk.png")

print("Environment and test blueprints are 100% ready!")


## Step 3: Prepare YOLO Segmentation Dataset
Formats floor plan ground truth into YOLOv8 polygon instance segmentation (`data.yaml`).

In [ ]:
from floorplan_reader.segmentation.dataset_exporter import YoloDatasetExporter
from floorplan_reader.segmentation.yolo_segmenter import YoloFloorPlanSegmenter

dataset_dir = Path("data/yolo_dataset")
exporter = YoloDatasetExporter(dataset_dir)
exporter.initialize_directories()
yaml_path = exporter.write_data_yaml()

# Export training samples
segmenter = YoloFloorPlanSegmenter()
for sample_name in ["sample_master_suite.png", "sample_1bhk.png"]:
    p = Path(f"data/test_samples/{sample_name}")
    if p.exists():
        res = segmenter.analyze(p)
        exporter.add_sample(p, res.to_clean_dict(), split="train", sample_id=p.stem)
        exporter.add_sample(p, res.to_clean_dict(), split="val", sample_id=f"{p.stem}_val")

print(f"YOLO segmentation dataset created at: {yaml_path}")
print(yaml_path.read_text())


## Step 4: Train YOLOv8 Instance Segmentation Model
Trains `yolov8n-seg.pt` on the prepared floor plan dataset.

In [ ]:
from ultralytics import YOLO

# Load pretrained nano segmentation weights
model = YOLO("yolov8n-seg.pt")

# Train on dataset (15-25 epochs on Colab T4 GPU)
epochs = 20 if torch.cuda.is_available() else 3
results = model.train(
    data=str(yaml_path),
    epochs=epochs,
    imgsz=640,
    batch=8 if torch.cuda.is_available() else 2,
    device=0 if torch.cuda.is_available() else "cpu",
    plots=True,
    verbose=True
)

best_weights = Path(model.trainer.save_dir) / "weights" / "best.pt"
print(f"Training complete! Best weights saved to: {best_weights}")


## Step 5: Run Architectural Analysis (Primary & Secondary Goals)
Executes room segmentation, dimension extraction, and door/window detection.

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt
from floorplan_reader.segmentation.yolo_segmenter import YoloFloorPlanSegmenter
from floorplan_reader.visualization.segmentation_visualizer import draw_styled_segmentation_overlay

# Initialize segmenter with trained weights (and automatic wall-snapping fallback)
analyzer = YoloFloorPlanSegmenter(
    weights_path=best_weights if best_weights.exists() else None,
    confidence_threshold=0.25
)

test_blueprints = [
    "data/test_samples/sample_master_suite.png",
    "data/test_samples/sample_1bhk.png"
]

for img_file in test_blueprints:
    if not Path(img_file).exists():
        continue
    print("\n=========================================")
    print(f"Analyzing: {img_file}")
    print("=========================================")
    
    analysis = analyzer.analyze(img_file)
    
    print("PRIMARY GOAL (Rooms & Real Dimensions):")
    for r in analysis.rooms:
        dim_str = f"{r.real_length:.2f}m x {r.real_width:.2f}m" if r.real_length else "N/A"
        area_str = f"{r.real_length * r.real_width:.2f} sq m" if r.real_length else "N/A"
        print(f"  • {r.name.upper():16}: {dim_str} | Area: {area_str} | Box: {r.box_2d.to_list()}")
        
    print("\nSECONDARY GOAL (Doors & Windows):")
    print(f"  • Doors detected:   {len(analysis.doors)}")
    for d in analysis.doors:
        print(f"     - Door {d.id} ({d.door_type}) at {d.box_2d.to_list()}")
    print(f"  • Windows detected: {len(analysis.windows)}")
    for w in analysis.windows:
        print(f"     - Window {w.id} ({w.window_type}) at {w.box_2d.to_list()}")
        
    # Render and display visual overlay
    out_overlay_path = f"data/test_samples/{Path(img_file).stem}_result_overlay.png"
    vis_img = draw_styled_segmentation_overlay(img_file, analysis, output_path=out_overlay_path)
    
    fig, ax = plt.subplots(figsize=(10, 8))
    ax.imshow(vis_img)
    ax.axis("off")
    ax.set_title(f"Detected Rooms, Dimensions, Doors & Windows: {Path(img_file).name}", fontsize=14)
    plt.show()


## Step 6: Interactive Upload (Analyze Your Own Floor Plan)
Upload your own blueprint image (.png, .jpg, or .svg) to analyze it directly.

In [ ]:
from google.colab import files
import io
from pathlib import Path
import matplotlib.pyplot as plt
from floorplan_reader.segmentation.yolo_segmenter import YoloFloorPlanSegmenter
from floorplan_reader.visualization.segmentation_visualizer import draw_styled_segmentation_overlay

# Ensure analyzer is initialized
if "analyzer" not in globals():
    best_w = Path("runs/segment/train/weights/best.pt")
    analyzer = YoloFloorPlanSegmenter(weights_path=best_w if best_w.exists() else None)

print("Upload a floor plan image (PNG, JPG, or SVG):")
uploaded = files.upload()

for filename in uploaded.keys():
    print(f"\nProcessing uploaded drawing: {filename}...")
    res = analyzer.analyze(filename)
    
    print(f"\nDetected {len(res.rooms)} Rooms:")
    for r in res.rooms:
        d_str = f"{r.real_length:.2f}m x {r.real_width:.2f}m ({r.real_length * r.real_width:.2f} sq m)" if r.real_length else "N/A"
        print(f"  • {r.name}: {d_str}")
        
    print(f"\nDetected {len(res.doors)} Doors & {len(res.windows)} Windows.")
    
    vis = draw_styled_segmentation_overlay(filename, res, output_path=f"annotated_{filename}.png")
    
    fig, ax = plt.subplots(figsize=(10, 8))
    ax.imshow(vis)
    ax.axis("off")
    ax.set_title(f"Analysis: {filename}", fontsize=14)
    plt.show()
